# 02 · RFM 用户分群

**目标**：把 5,939 个客户切成可操作的运营群组（冠军 / 忠诚 / 流失预警 / 已流失高价值 ...）。

**RFM 定义**：
- **R**ecency：距观察点的天数（越小越好）
- **F**requency：不同订单数（`Invoice` nunique）
- **M**onetary：净成交额（含退货扣减）

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from rfm import compute_rfm, SEGMENT_MAP
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df = pd.read_parquet('../data/processed/transactions.parquet')
rfm = compute_rfm(df)
rfm.head()

,customer_id,recency,frequency,monetary,R_score,F_score,M_score,segment
1,12347,2,8,5633.32,4,3,4,忠诚用户
2,12348,75,5,2019.40,3,3,3,稳定用户
3,12349,19,5,4404.54,4,3,4,忠诚用户
4,12350,310,1,334.40,2,1,1,低频新客
5,12351,375,1,300.93,2,1,1,低频新客


## 1. 各维度分布

In [2]:
print(rfm[['recency', 'frequency', 'monetary']].describe().round(1))

       recency  frequency  monetary
count   5842.0     5842.0    5842.0
mean     197.4        7.6    2865.9
std      207.9       16.1   14079.3
min        1.0        1.0       0.0
25%       24.0        2.0     340.6
50%       92.0        4.0     876.9
75%      376.0        8.0    2235.2
max      739.0      510.0  598215.2


In [3]:
fig = px.histogram(rfm, x='recency', nbins=60, title='Recency 分布 (距观察点天数)')
fig.update_layout(height=320)
fig.show()
fig = px.histogram(rfm[rfm['frequency'] <= 30], x='frequency', nbins=30, title='Frequency 分布（截断 30）')
fig.update_layout(height=320)
fig.show()
fig = px.histogram(np.log10(rfm[rfm['monetary'] > 0]['monetary']), nbins=60, title='Monetary 分布 (log10 £)')
fig.update_layout(height=320, xaxis_title='log10(总消费金额)')
fig.show()

## 2. 分群统计

In [4]:
seg_stats = (rfm.groupby('segment')
             .agg(customers=('customer_id', 'count'),
                  avg_recency=('recency', 'mean'),
                  avg_frequency=('frequency', 'mean'),
                  avg_monetary=('monetary', 'mean'),
                  total_monetary=('monetary', 'sum'))
             .sort_values('total_monetary', ascending=False))
seg_stats['customer_share'] = seg_stats['customers'] / seg_stats['customers'].sum()
seg_stats['gmv_share'] = seg_stats['total_monetary'] / seg_stats['total_monetary'].sum()
seg_stats.style.format({
    'avg_recency': '{:.0f} 天',
    'avg_frequency': '{:.1f}',
    'avg_monetary': '£{:,.0f}',
    'total_monetary': '£{:,.0f}',
    'customer_share': '{:.1%}',
    'gmv_share': '{:.1%}',
})

,customers,avg_recency,avg_frequency,avg_monetary,total_monetary,customer_share,gmv_share
segment,,,,,,,
冠军用户,747,10 天,27.7,"£12,543","£9,369,264",12.8%,56.0%
忠诚用户,846,32 天,10.9,"£3,541","£2,995,383",14.5%,17.9%
流失预警,214,185 天,14.9,"£4,714","£1,008,699",3.7%,6.0%
稳定用户,419,53 天,5.5,"£1,704","£713,981",7.2%,4.3%
需要挽回,455,211 天,5.4,"£1,526","£694,204",7.8%,4.1%
一般用户,764,153 天,2.6,£749,"£572,281",13.1%,3.4%
已流失,652,480 天,3.2,£837,"£545,650",11.2%,3.3%
沉睡用户,753,540 天,1.0,£327,"£245,930",12.9%,1.5%
潜力用户,239,12 天,2.5,£796,"£190,301",4.1%,1.1%


> **关键洞察**：
> - **冠军用户 + 忠诚用户**：以 **27.3%** 的人数占比贡献 **73.9%** 的 GMV —— 接近 80/20 的集中度
> - **已流失高价值**这一小撮（45 人）人均消费惊人，是挽回动作的最高 ROI 目标
> - **流失预警**（近期未回访但历史高价值）应立即触发召回

## 3. 分群可视化 — 气泡图

In [5]:
bubble = seg_stats.reset_index()
fig = px.scatter(bubble, x='avg_recency', y='avg_frequency',
                 size='total_monetary', color='segment',
                 text='segment', size_max=60,
                 title='RFM 分群 · 气泡大小 = 总 GMV')
fig.update_traces(textposition='top center')
fig.update_layout(height=520, xaxis_title='平均 Recency (天, 越小越近)',
                  yaxis_title='平均 Frequency (订单数)')
fig.show()

## 4. K-Means 交叉验证 — 规则分群 vs 数据驱动

In [6]:
X = rfm[['recency', 'frequency', 'monetary']].copy()
X['monetary'] = np.log1p(X['monetary'])
X_scaled = StandardScaler().fit_transform(X)

inertias = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    inertias.append(km.inertia_)
fig = px.line(x=list(range(2, 9)), y=inertias, markers=True,
              title='K-Means 肘部法', labels={'x': 'k', 'y': 'inertia'})
fig.update_layout(height=320)
fig.show()

In [7]:
km = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_scaled)
rfm['kmeans_cluster'] = km.labels_
cross = pd.crosstab(rfm['segment'], rfm['kmeans_cluster'])
cross

kmeans_cluster,0,1,2,3
segment,,,,
一般用户,135,8,621,0
低频新客,177,2,179,0
冠军用户,0,663,66,18
已流失,651,1,0,0
已流失高价值,36,9,0,0
忠诚用户,0,451,394,1
新客,1,2,347,0
沉睡用户,752,1,0,0
流失预警,9,132,72,1


> K=4 时 inertia 拐点明显，与 RFM 规则分群的大类（活跃高价值 / 活跃低价值 / 流失高价值 / 流失低价值）吻合，**规则分群的运营语义更强，建议作为对外沟通的主口径；K-Means 作为数据驱动的验证。**

## 5. 导出给 03 建模使用

In [8]:
rfm.to_parquet('../data/processed/rfm.parquet', index=False)
print(f'✓ 已保存 {len(rfm):,} 条 RFM 记录')

✓ 已保存 5,842 条 RFM 记录


---

下一步 → [03_repurchase_prediction.ipynb](03_repurchase_prediction.ipynb)